# 04. Adaptive RAG: 질문 유형에 따른 적응형 검색

> 모든 질문이 같은 RAG 파이프라인을 거칠 필요는 없어요. Query Router로 vectorstore · web_search · LLM 단독을 골라 보내는 적응형 RAG를 구현해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **Query Router**를 구현하여 질문을 `vectorstore` 또는 `web_search`로 자동 분류할 수 있어요
2. **5개 핵심 노드**(route_question, retrieve, grade_documents, generate, transform_query + web_search)로 구성된 Adaptive RAG 그래프를 만들 수 있어요
3. **3단계 조건부 라우팅**(질문 라우팅 → 문서 평가 → 환각/관련성 검증)으로 검색 전략을 동적으로 선택할 수 있어요
4. **Hallucination Grader**와 **Answer Grader**를 조합하여 생성된 답변의 품질을 자동 검증할 수 있어요
5. CRAG, Self-RAG와 Adaptive RAG의 차이를 설명하고, 언제 어떤 방식을 선택할지 판단할 수 있어요

## 사전 지식

- 이전 노트북 `03-CRAG-Self-RAG.ipynb`에서 배운 GradeDocuments, Hallucination Grader, Query Rewriter 패턴
- LangGraph의 StateGraph, 조건부 엣지, MemorySaver 기본 사용법
- Pydantic BaseModel과 `with_structured_output()`으로 구조화된 출력 다루기

## Adaptive RAG란?

**Adaptive RAG**는 [논문(arXiv:2403.14403)](https://arxiv.org/abs/2403.14403)에서 제안된 전략으로, 질문의 성격에 따라 다른 검색 전략을 선택해요.

> 🔑 **핵심 개념**: Adaptive RAG는 **대형병원의 접수 시스템**과 같아요. 환자가 들어오면 접수 창구(Query Router)에서 증상을 듣고 "내과로 가세요" 또는 "외과로 가세요"를 먼저 결정해요. CRAG/Self-RAG가 "일단 내과에 가봤다가 아니면 외과로 가는" 방식이라면, Adaptive RAG는 **처음부터 맞는 곳으로 안내**해요.

### 왜 이전 방식으로는 부족한가요?

CRAG와 Self-RAG는 항상 벡터스토어 검색부터 시작해요. 하지만 "2024년 노벨상 수상자"처럼 PDF에 없는 것이 명백한 질문도 일단 벡터 검색을 하고 → 실패하고 → 그제서야 웹 검색으로 가요. 이건 시간과 비용 낭비예요.

| 방식 | 특징 | 한계 |
|------|------|------|
| **Naive RAG** | 항상 벡터 검색 | 최신 정보 못 찾음 |
| **CRAG** | 문서 없으면 웹 검색으로 보강 | 처음부터 라우팅 없음 (시간 낭비) |
| **Self-RAG** | 생성 품질 자체 검증 | 웹 검색 없음 |
| **Adaptive RAG** | 질문 분류 → 최적 경로 선택 | 세 방식의 장점 결합 |

### Adaptive RAG 전체 아키텍처

```mermaid
flowchart TD
    A([START]) --> B{route_question<br/>질문 라우팅}
    B -- "vectorstore" --> C[retrieve<br/>벡터 검색]
    B -- "web_search" --> D[web_search<br/>웹 검색]
    D --> E[generate<br/>답변 생성]
    C --> F[grade_documents<br/>문서 관련성 평가]
    F --> G{decide_to_generate<br/>평가 결과?}
    G -- "관련 문서 있음" --> E
    G -- "관련 문서 없음" --> H[transform_query<br/>질문 재작성]
    H --> C
    E --> I{hallucination_check<br/>품질 검증}
    I -- "환각 감지" --> E
    I -- "질문 미해결" --> H
    I -- "좋은 답변" --> J([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,J terminal
    class C,D,E,H process
    class B,G,F,I decision
```

### Adaptive RAG = CRAG + Self-RAG + Query Router

| 구성 요소 | 출처 | 역할 |
|----------|------|------|
| **Query Router** | Adaptive RAG 고유 | 질문 → vectorstore/web_search 사전 분류 |
| **Retrieval Grader** | CRAG에서 가져옴 | 문서 관련성 평가 (yes/no) |
| **RAG Chain** | 공통 | 문서 기반 답변 생성 |
| **Hallucination Grader** | Self-RAG에서 가져옴 | 답변이 문서에 근거하는지 검증 |
| **Answer Grader** | Self-RAG에서 가져옴 | 답변이 질문을 해결하는지 검증 |
| **Query Rewriter** | CRAG/Self-RAG 공통 | 의미적 의도 추출, 검색 최적화 |

> 🎯 **강의 포인트**: Adaptive RAG를 이해하면 이전 노트북의 모든 기법이 어떻게 하나로 합쳐지는지 볼 수 있어요. CRAG의 "웹 검색 보강" + Self-RAG의 "생성 품질 검증" + 새로운 "사전 라우팅"이 결합된 형태예요.

## 1. 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 읽어와요
from dotenv import load_dotenv

# API 키 정보 로드 (OPENAI_API_KEY, TAVILY_API_KEY 등)
load_dotenv(override=True)

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정
# ---------------------------------------------------
# LangSmith를 사용하면 그래프 실행 과정을 시각적으로 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-RAG-Adaptive"

## 2. 기본 모델 및 Retriever 설정

Adaptive RAG는 5가지 평가기를 모두 사용해요. 각 평가기에 동일한 `gpt-4o-mini`를 재사용하되, `temperature=0`으로 고정하여 일관된 평가 결과를 얻어요.

> 💡 **실무 팁**: 평가기용 모델은 `temperature=0`이 필수예요. 평가 결과가 랜덤하게 바뀌면 시스템이 불안정해져요. 반면 답변 생성용 모델은 약간의 창의성을 위해 `temperature=0.1` 정도를 사용하기도 해요.

In [ ]:
# ---------------------------------------------------
# 기본 모델 초기화
# ---------------------------------------------------
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델 옵션:
#   - "anthropic:claude-sonnet-4-5" (더 강력한 추론)
#   - "ollama:llama3" (로컬 실행)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 기본 모델 준비 완료 (gpt-4o-mini, temperature=0)

In [ ]:
# ---------------------------------------------------
# PDF 로더, 임베딩, 벡터스토어 설정
# ---------------------------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# PDF 파일 로드 (SPRI AI Brief 2023년 12월호)
# 이 문서에는 삼성 가우스, 앤스로픽 투자 등 2023년 AI 동향이 담겨 있어요
PDF_PATH = "data/SPRI_AI_Brief_2023년12월호_F.pdf"

# 페이지 단위로 문서 로드
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"총 {len(pages)}페이지 로드 완료")

# 청크로 분할: 500자 단위, 50자 오버랩
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs = splitter.split_documents(pages)
print(f"총 {len(docs)}개 청크 생성")

# OpenAI 임베딩 + FAISS 벡터스토어 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

# Retriever 생성 (유사도 기반, 상위 4개 문서 반환)
pdf_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
# Retriever 준비 완료

In [ ]:
# ---------------------------------------------------
# RAG 체인 구성 (답변 생성용)
# ---------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# RAG 프롬프트 템플릿 정의
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the question based on the provided context.
If the context doesn't contain enough information, say so clearly.
Always cite your sources by mentioning the document name and page number."""),
    ("human", """Context: {context}\n\nQuestion: {question}\n\nAnswer in Korean:""")
])


def format_docs(docs):
    """문서 리스트를 프롬프트 입력 형식으로 포맷팅해요."""
    return "\n\n".join(
        f"<document>\n<content>{doc.page_content}</content>\n"
        f"<source>{doc.metadata.get('source', 'unknown')}</source>\n"
        f"<page>{doc.metadata.get('page', 0) + 1}</page>\n</document>"
        for doc in docs
    )


# RAG 체인: 프롬프트 → LLM → 텍스트 파서
rag_chain = RAG_PROMPT | llm | StrOutputParser()
# RAG 체인 준비 완료

## 3. Query Router 구성

Adaptive RAG에서 가장 중요한 첫 번째 단계예요. 질문을 분석하여 벡터스토어 검색과 웹 검색 중 어느 쪽이 적합한지 결정해요.

> 🔑 **핵심 개념**: `Literal["vectorstore", "web_search"]`로 타입을 제한하면 LLM이 이 두 값 중 하나만 반환해요. `with_structured_output(RouteQuery)`가 이를 강제하므로, 파싱 에러 없이 항상 유효한 라우팅 결정을 받아요.

> ⚠️ **자주 하는 실수**: 시스템 프롬프트에서 벡터스토어가 어떤 내용을 담고 있는지 명확히 알려줘야 해요. 라우터가 맥락 없이 판단하면 잘못된 경로로 가게 돼요.

In [ ]:
# ---------------------------------------------------
# Query Router: 질문 라우팅 컴포넌트
# ---------------------------------------------------
# 역할: 질문을 분석하여 vectorstore 또는 web_search로 분류해요
from typing import Literal
from pydantic import BaseModel, Field


# 라우팅 결정을 위한 Pydantic 모델
class RouteQuery(BaseModel):
    """질문을 가장 적합한 데이터 소스로 라우팅하는 모델이에요."""

    # Literal 타입으로 두 값만 허용해요
    datasource: Literal["vectorstore", "web_search"] = Field(
        ...,
        description="질문에 따라 vectorstore 또는 web_search를 선택해요.",
    )


# RouteQuery 구조로만 응답하도록 강제
structured_llm_router = llm.with_structured_output(RouteQuery)

# 라우터 시스템 프롬프트
# 벡터스토어 내용을 명확히 설명해야 올바른 라우팅이 돼요
router_system = """You are an expert at routing a user question to a vectorstore or web search.
The vectorstore contains documents related to the SPRI AI Brief Report (December 2023),
covering topics like Samsung Gauss, Anthropic investments, OpenAI updates, and AI industry news from 2023.
Use the vectorstore for questions about these specific 2023 AI topics.
For questions about recent events (2024 onwards), current news, or topics not in the AI brief, use web search."""

# 라우팅 프롬프트 템플릿
route_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system),
    ("human", "{question}"),
])

# 질문 라우터 체인
question_router = route_prompt | structured_llm_router
# Query Router 준비 완료

In [ ]:
# ---------------------------------------------------
# Query Router 테스트
# ---------------------------------------------------
# 두 종류의 질문으로 라우팅 결과를 확인해요

# 테스트 1: 벡터스토어 문서에 있는 정보
q1 = "AI Brief 에서 삼성전자가 만든 생성형 AI 의 이름은?"
result1 = question_router.invoke({"question": q1})
print(f"질문: {q1}")
print(f"라우팅 결과: {result1.datasource}")

print()

# 테스트 2: 최신 정보가 필요한 질문
q2 = "2025년 최신 AI 모델 현황은?"
result2 = question_router.invoke({"question": q2})
print(f"질문: {q2}")
print(f"라우팅 결과: {result2.datasource}")

## 4. 문서 관련성 평가기 (Retrieval Grader)

벡터스토어에서 검색된 문서가 실제로 질문과 관련이 있는지 검증해요. 이 단계에서 관련 없는 문서를 필터링하고, 관련 문서가 없으면 쿼리 재작성 경로로 보내요.

> 💡 **실무 팁**: 관련성 평가는 엄격하게 하지 않아도 돼요. "키워드나 의미적 유사성이 있으면 yes"라는 기준으로 설정하면, 불필요하게 쿼리를 재작성하는 빈도를 줄일 수 있어요.

In [ ]:
# ---------------------------------------------------
# 문서 관련성 평가기 (Retrieval Grader)
# ---------------------------------------------------
# 역할: 검색된 각 문서가 질문과 관련이 있는지 yes/no로 평가해요


class GradeDocuments(BaseModel):
    """검색된 문서의 관련성을 평가하는 이진 점수예요."""

    binary_score: str = Field(
        description="문서가 질문과 관련이 있으면 'yes', 없으면 'no'"
    )


# 구조화된 출력을 사용하는 평가기 LLM
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# 관련성 평가 프롬프트: 엄격하지 않게 설정해요 (키워드 or 의미적 유사성)
grade_system = """You are a grader assessing relevance of a retrieved document to a user question.
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.
It does not need to be a stringent test. The goal is to filter out clearly erroneous retrievals.
Give a binary score 'yes' or 'no' to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", grade_system),
    ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
])

# 관련성 평가기 체인
retrieval_grader = grade_prompt | structured_llm_grader
# Retrieval Grader 준비 완료

In [ ]:
# ---------------------------------------------------
# Retrieval Grader 테스트
# ---------------------------------------------------
# 실제 검색 결과에 대한 관련성 평가를 확인해요

test_question = "삼성전자가 개발한 생성형 AI의 이름은?"
test_docs = pdf_retriever.invoke(test_question)

print(f"질문: {test_question}")
print(f"검색된 문서 수: {len(test_docs)}개")
print()

# 각 문서의 관련성 평가
for i, doc in enumerate(test_docs):
    score = retrieval_grader.invoke({
        "question": test_question,
        "document": doc.page_content
    })
    # 문서 내용 일부와 평가 결과 출력
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"문서 {i+1}: [{score.binary_score}] {preview}...")

## 5. Hallucination Grader와 Answer Grader 구성

Self-RAG에서 배운 두 평가기를 그대로 사용해요. Adaptive RAG에서는 이 두 평가기를 조합하여 `hallucination_check` 조건부 엣지 하나로 관리해요.

> 🎯 **강의 포인트**: 두 평가기가 순차적으로 동작하는 것을 설명해요:
> 1. **Hallucination Grader**: "이 답변이 문서에 근거하는가?" → No이면 바로 재생성
> 2. **Answer Grader**: "이 답변이 질문을 해결하는가?" → No이면 질문 재작성
> 두 평가를 모두 통과해야만 END로 진행해요.

In [ ]:
# ---------------------------------------------------
# Hallucination Grader: 환각 검증 평가기
# ---------------------------------------------------
# 역할: 생성된 답변이 검색된 문서에 근거하는지 검증해요
# yes = 문서에 근거함 (환각 없음)
# no  = 문서와 무관한 내용 (환각 가능성, 재생성 필요)


class GradeHallucinations(BaseModel):
    """생성된 답변의 환각 여부를 평가하는 이진 점수예요."""

    binary_score: str = Field(
        description="답변이 문서에 근거하면 'yes', 아니면 'no'"
    )


# 환각 평가 전용 LLM
hallucination_grader_llm = llm.with_structured_output(GradeHallucinations)

hallucination_system = """You are a grader assessing whether an LLM generation is grounded in / 
supported by a set of retrieved facts.
Give a binary score 'yes' or 'no'.
'Yes' means that the answer is grounded in / supported by the set of facts."""

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", hallucination_system),
    ("human", "Set of facts:\n\n{documents}\n\nLLM generation: {generation}"),
])

hallucination_grader = hallucination_prompt | hallucination_grader_llm
# Hallucination Grader 준비 완료

In [ ]:
# ---------------------------------------------------
# Answer Grader: 답변 관련성 평가기
# ---------------------------------------------------
# 역할: 생성된 답변이 실제로 질문을 해결하는지 검증해요
# yes = 질문을 해결함 → END
# no  = 질문에 답하지 않음 → 질문 재작성 필요


class GradeAnswer(BaseModel):
    """답변이 질문을 해결하는지 평가하는 이진 점수예요."""

    binary_score: str = Field(
        description="답변이 질문을 해결하면 'yes', 아니면 'no'"
    )


# 답변 관련성 평가 전용 LLM
answer_grader_llm = llm.with_structured_output(GradeAnswer)

answer_system = """You are a grader assessing whether an answer addresses / resolves a question.
Give a binary score 'yes' or 'no'.
'Yes' means that the answer resolves the question."""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", answer_system),
    ("human", "User question:\n\n{question}\n\nLLM generation: {generation}"),
])

answer_grader = answer_prompt | answer_grader_llm
# Answer Grader 준비 완료

## 6. Query Rewriter와 Web Search Tool 구성

벡터 검색에 실패하거나 답변 품질이 낮을 때 사용하는 두 가지 도구예요.

In [ ]:
# ---------------------------------------------------
# Query Rewriter: 질문 재작성기
# ---------------------------------------------------
# 역할: 검색 실패 또는 품질 저하 시 의미적 의도를 추출하여 더 나은 질문 생성

rewrite_system = """You are a question re-writer that converts an input question to a better version 
that is optimized for vectorstore retrieval.
Look at the input and try to reason about the underlying semantic intent / meaning.
Output only the improved question, no explanation."""

re_write_prompt = ChatPromptTemplate.from_messages([
    ("system", rewrite_system),
    ("human", "Here is the initial question:\n\n{question}\n\nFormulate an improved question."),
])

# 질문 재작성기: 프롬프트 → LLM → 텍스트
question_rewriter = re_write_prompt | llm | StrOutputParser()

# 테스트: 간단한 질문이 어떻게 개선되는지 확인
test_q = "삼성전자 AI"
rewritten = question_rewriter.invoke({"question": test_q})
print(f"원본 질문: {test_q}")
print(f"재작성 질문: {rewritten}")

In [ ]:
# ---------------------------------------------------
# 웹 검색 도구 (Tavily Search)
# ---------------------------------------------------
# 역할: 벡터스토어에 없는 최신 정보를 웹에서 검색해요
# 사전 준비: .env 파일에 TAVILY_API_KEY 설정 필요
# 발급 주소: https://tavily.com
from langchain_tavily import TavilySearch

# 웹 검색 도구 생성 (최대 3개 결과)
web_search_tool = TavilySearch(max_results=3)

# 테스트: 웹 검색 결과 확인
# TavilySearch.invoke()는 {"results": [...], ...} 딕셔너리를 반환해요
response = web_search_tool.invoke({"query": "2024 AI 최신 동향"})
results = response.get("results", [])
print(f"웹 검색 결과 수: {len(results)}개")
if results:
    content = results[0].get('content', str(results[0]))
    print(f"첫 번째 결과 미리보기: {content[:100]}...")


## 7. Adaptive RAG 그래프 구성

5개의 컴포넌트를 모두 준비했어요. 이제 LangGraph로 연결해볼게요.

### 그래프 상태(State) 정의

Adaptive RAG의 상태는 CRAG/Self-RAG보다 단순해요. `question`, `generation`, `documents` 세 가지만 있으면 돼요.

In [ ]:
# ---------------------------------------------------
# Adaptive RAG 상태(State) 정의
# ---------------------------------------------------
from typing import Annotated, List
from typing_extensions import TypedDict


class GraphState(TypedDict):
    """Adaptive RAG 그래프의 상태 정의예요.

    Attributes:
        question: 사용자 질문 (질문 재작성 시 업데이트됨)
        generation: LLM이 생성한 답변
        documents: 검색된 문서 리스트
    """

    question: Annotated[str, "사용자 질문"]
    generation: Annotated[str, "LLM 생성 답변"]
    documents: Annotated[List, "검색된 문서 리스트"]


# GraphState 정의 완료

### 노드 함수 정의

5개의 실행 노드와 3개의 조건부 엣지 함수를 정의해요.

In [ ]:
# ---------------------------------------------------
# 노드 1: retrieve - 벡터 검색
# ---------------------------------------------------
# 역할: 질문을 바탕으로 벡터스토어에서 관련 문서를 검색해요


def retrieve(state: GraphState) -> GraphState:
    """벡터스토어에서 문서를 검색하는 노드예요."""
    # ==== [RETRIEVE] ====
    question = state["question"]

    # FAISS 벡터스토어에서 유사도 검색
    documents = pdf_retriever.invoke(question)
    print(f"  검색된 문서 수: {len(documents)}개")
    return {"documents": documents}

In [ ]:
# ---------------------------------------------------
# 노드 2: web_search - 웹 검색
# ---------------------------------------------------
# 역할: Tavily API로 최신 정보를 검색하여 Document 형식으로 반환해요
from langchain_core.documents import Document


def web_search(state: GraphState) -> GraphState:
    """웹 검색을 수행하는 노드예요."""
    # ==== [WEB SEARCH] ====
    question = state["question"]

    # Tavily로 웹 검색
    # TavilySearch.invoke()는 {"results": [...], ...} 딕셔너리를 반환해요
    response = web_search_tool.invoke({"query": question})
    results = response.get("results", [])

    # 검색 결과를 Document 형식으로 변환
    web_docs = [
        Document(
            page_content=r["content"],
            metadata={"source": r["url"], "type": "web_search"}
        )
        for r in results
    ]
    print(f"  웹 검색 결과: {len(web_docs)}개")
    return {"documents": web_docs}

In [ ]:
# ---------------------------------------------------
# 노드 3: grade_documents - 문서 관련성 평가
# ---------------------------------------------------
# 역할: 검색된 문서 중 질문과 관련 없는 문서를 필터링해요


def grade_documents(state: GraphState) -> GraphState:
    """검색된 문서의 관련성을 평가하고 필터링하는 노드예요."""
    # ==== [GRADE DOCUMENTS] ====
    question = state["question"]
    documents = state["documents"]

    # 각 문서에 대한 관련성 평가
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke({
            "question": question,
            "document": d.page_content
        })
        if score.binary_score == "yes":
            #   ✓ 관련 문서 - 유지
            filtered_docs.append(d)
        else:
            #   ✗ 관련 없는 문서 - 필터링
            pass

    print(f"  필터링 후 문서 수: {len(filtered_docs)}개")
    return {"documents": filtered_docs}

In [ ]:
# ---------------------------------------------------
# 노드 4: generate - 답변 생성
# ---------------------------------------------------
# 역할: 검색된 문서를 기반으로 최종 답변을 생성해요


def generate(state: GraphState) -> GraphState:
    """문서를 기반으로 답변을 생성하는 노드예요."""
    # ==== [GENERATE] ====
    question = state["question"]
    documents = state["documents"]

    # 문서가 Document 객체인지 확인 후 포맷팅
    if documents and hasattr(documents[0], "page_content"):
        context = format_docs(documents)
    else:
        context = str(documents)

    # RAG 체인으로 답변 생성
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"generation": generation}

In [ ]:
# ---------------------------------------------------
# 노드 5: transform_query - 질문 재작성
# ---------------------------------------------------
# 역할: 벡터 검색이 실패했거나 답변 품질이 낮을 때 질문을 개선해요


def transform_query(state: GraphState) -> GraphState:
    """질문을 재작성하는 노드예요."""
    # ==== [TRANSFORM QUERY] ====
    question = state["question"]

    # 의미적 의도를 추출하여 더 나은 질문으로 변환
    better_question = question_rewriter.invoke({"question": question})
    print(f"  원본: {question}")
    print(f"  재작성: {better_question}")
    return {"question": better_question}

### 조건부 엣지 함수 정의

Adaptive RAG의 3단계 조건부 라우팅을 구현해요.

> 🎯 **강의 포인트**: 세 개의 조건부 엣지가 서로 다른 시점에 다른 판단을 해요:
> - **route_question**: 처음 진입 시 소스 결정 (web vs vectorstore)
> - **decide_to_generate**: 문서 평가 후 진행 여부 결정 (generate vs rewrite)
> - **hallucination_check**: 생성 후 품질 결정 (end vs retry vs rewrite)

In [ ]:
# ---------------------------------------------------
# 조건부 엣지 1: route_question - 최초 라우팅
# ---------------------------------------------------
# 역할: 질문의 성격에 따라 web_search 또는 vectorstore 경로를 선택해요


def route_question(state: GraphState) -> str:
    """질문을 분석하여 데이터 소스를 결정하는 라우팅 함수예요.

    Returns:
        'web_search': 최신 정보가 필요한 질문
        'vectorstore': 문서에 있는 정보에 대한 질문
    """
    # ==== [ROUTE QUESTION] ====
    question = state["question"]

    # RouteQuery 모델로 라우팅 결정
    source = question_router.invoke({"question": question})

    if source.datasource == "web_search":
        #   결정: 웹 검색으로 라우팅
        return "web_search"
    else:
        #   결정: 벡터스토어로 라우팅
        return "vectorstore"

In [ ]:
# ---------------------------------------------------
# 조건부 엣지 2: decide_to_generate - 문서 평가 후 분기
# ---------------------------------------------------
# 역할: 필터링된 문서가 있으면 생성, 없으면 질문 재작성


def decide_to_generate(state: GraphState) -> str:
    """문서 관련성 평가 결과에 따라 다음 단계를 결정해요.

    Returns:
        'transform_query': 관련 문서가 없어 재작성 필요
        'generate': 관련 문서가 있어 답변 생성 가능
    """
    # ==== [DECIDE TO GENERATE] ====
    filtered_documents = state["documents"]

    if not filtered_documents:
        # 관련 문서가 하나도 없음 → 질문 재작성 후 재검색
        #   결정: 질문 재작성 (관련 문서 없음)
        return "transform_query"
    else:
        # 관련 문서 있음 → 바로 생성
        #   결정: 답변 생성 (관련 문서 있음)
        return "generate"

In [ ]:
# ---------------------------------------------------
# 조건부 엣지 3: hallucination_check - 생성 품질 검증
# ---------------------------------------------------
# 역할: 환각 검증 → 답변 관련성 검증의 2단계 품질 체크
# 이 함수 하나에서 hallucination_grader + answer_grader를 모두 실행해요


def hallucination_check(state: GraphState) -> str:
    """생성된 답변의 품질을 2단계로 검증하는 조건부 엣지예요.

    단계 1: 환각 검증 (문서에 근거하는가?)
    단계 2: 답변 관련성 검증 (질문을 해결하는가?)

    Returns:
        'relevant'    : 두 검증 모두 통과 → END
        'not relevant': 답변이 질문을 해결하지 못함 → transform_query
        'hallucination': 환각 감지 → 다시 generate
    """
    # ==== [HALLUCINATION CHECK] ====
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]

    # 문서 텍스트 추출 (평가기 입력용)
    docs_text = "\n".join([d.page_content for d in documents]) if documents else ""

    # 단계 1: 환각 검증
    hallucination_score = hallucination_grader.invoke({
        "documents": docs_text,
        "generation": generation
    })

    if hallucination_score.binary_score == "yes":
        #   ✓ 환각 없음 (문서에 근거)

        # 단계 2: 답변 관련성 검증
        answer_score = answer_grader.invoke({
            "question": question,
            "generation": generation
        })

        if answer_score.binary_score == "yes":
            #   ✓ 질문 해결됨 → 완료
            return "relevant"
        else:
            #   ✗ 질문 미해결 → 질문 재작성
            return "not relevant"
    else:
        #   ✗ 환각 감지 → 재생성
        return "hallucination"

## 8. 그래프 조립 및 컴파일

5개의 노드와 3개의 조건부 엣지를 연결하여 완전한 Adaptive RAG 그래프를 만들어요.

> 💡 **실무 팁**: `MemorySaver`를 체크포인터로 사용하면 대화 히스토리를 유지할 수 있어요. Adaptive RAG는 루프 구조가 있으므로 반드시 `recursion_limit`을 설정해야 해요.

In [ ]:
# ---------------------------------------------------
# Adaptive RAG 그래프 조립
# ---------------------------------------------------
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

# GraphState 기반 그래프 초기화
workflow = StateGraph(GraphState)

# ---- 노드 등록 ----
workflow.add_node("web_search", web_search)        # 웹 검색
workflow.add_node("retrieve", retrieve)              # 벡터 검색
workflow.add_node("grade_documents", grade_documents)  # 문서 관련성 평가
workflow.add_node("generate", generate)              # 답변 생성
workflow.add_node("transform_query", transform_query)  # 질문 재작성

# ---- 엣지 연결 ----

# 조건부 엣지 1: START → route_question → web_search 또는 retrieve
workflow.add_conditional_edges(
    START,
    route_question,
    {
        "web_search": "web_search",    # 최신 정보 → 웹 검색
        "vectorstore": "retrieve",     # 문서 정보 → 벡터 검색
    },
)

# 웹 검색 결과는 바로 생성으로 (평가 없이 직행)
workflow.add_edge("web_search", "generate")

# 벡터 검색 결과는 문서 평가를 거침
workflow.add_edge("retrieve", "grade_documents")

# 조건부 엣지 2: grade_documents → decide_to_generate → transform_query 또는 generate
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",  # 관련 문서 없음 → 재작성
        "generate": "generate",                # 관련 문서 있음 → 생성
    },
)

# 질문 재작성 후 다시 벡터 검색 (루프)
workflow.add_edge("transform_query", "retrieve")

# 조건부 엣지 3: generate → hallucination_check → END, transform_query, 또는 generate
workflow.add_conditional_edges(
    "generate",
    hallucination_check,
    {
        "hallucination": "generate",           # 환각 → 재생성
        "relevant": END,                       # 좋은 답변 → 종료
        "not relevant": "transform_query",    # 질문 미해결 → 재작성
    },
)

# ---- 그래프 컴파일 ----
# MemorySaver: 대화 히스토리 및 체크포인트 저장
adaptive_rag = workflow.compile(checkpointer=MemorySaver())

# Adaptive RAG 그래프 컴파일 완료
# 노드: web_search, retrieve, grade_documents, generate, transform_query
# 조건부 엣지: route_question, decide_to_generate, hallucination_check

In [ ]:
# ---------------------------------------------------
# 그래프 시각화
# ---------------------------------------------------
# 그래프 흐름: START → route_question → (web_search | retrieve) → ... → generate → hallucination_check → (END | generate | transform_query)
# route_question: 질문을 vectorstore 또는 web_search로 사전 라우팅해요
# web_search: 최신 정보를 Tavily로 검색해요 (문서 평가 없이 직행)
# retrieve: PDF 벡터 검색 후 grade_documents로 문서를 평가해요
# grade_documents → decide_to_generate: 관련 문서 유무에 따라 generate 또는 transform_query로 분기해요
# generate → hallucination_check: 환각 검증 + 답변 관련성 검증을 수행해요
# 두 가지 진입 경로(web_search vs vectorstore)와 품질 검증 루프를 확인해요
from IPython.display import Image, display

display(Image(adaptive_rag.get_graph().draw_mermaid_png()))

## 9. Adaptive RAG 실행 테스트

두 가지 경로를 모두 테스트해요:
1. **벡터스토어 경로**: PDF에 있는 정보 질문
2. **웹 검색 경로**: 최신 정보 질문

In [ ]:
# ---------------------------------------------------
# 테스트 1: 벡터스토어 경로 (PDF에 있는 정보)
# ---------------------------------------------------
import uuid
from langchain_core.runnables import RunnableConfig

# config 설정: recursion_limit으로 무한 루프 방지, thread_id로 대화 구분
config1 = RunnableConfig(
    recursion_limit=20,                            # 최대 재귀 횟수
    configurable={"thread_id": str(uuid.uuid4())}  # 고유 스레드 ID
)

# PDF 문서에 있는 정보 질문
inputs1 = {"question": "삼성전자가 개발한 생성형 AI의 이름은?"}

# ============================================================
print(f"질문: {inputs1['question']}")
# 경로 예상: START → retrieve → grade_documents → generate → END
# ============================================================

# 스트리밍 실행 (stream_mode='updates'로 각 노드의 출력만 확인)
final_answer = None
for chunk in adaptive_rag.stream(inputs1, config1, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- 노드: {node_name} ---")
        if "generation" in node_output:
            final_answer = node_output["generation"]

if final_answer:
    # ============================================================
    # 최종 답변:
    print(final_answer)

In [ ]:
# ---------------------------------------------------
# 테스트 2: 웹 검색 경로 (최신 정보)
# ---------------------------------------------------
# 이 질문은 2024년 이후의 최신 정보를 요구하므로
# route_question이 web_search로 라우팅해요
from langgraph.errors import GraphRecursionError

config2 = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": str(uuid.uuid4())}
)

# 최신 정보가 필요한 질문
inputs2 = {"question": "2024년 노벨 문학상 수상자는 누구인가요?"}

# ============================================================
print(f"질문: {inputs2['question']}")
# 경로 예상: START → web_search → generate → END
# ============================================================

# 웹 검색이 빈 결과를 반환하면 transform_query ↔ retrieve 루프가 발생할 수 있어
# GraphRecursionError를 명시적으로 처리해요 (실무 방어 패턴)
final_answer2 = None
try:
    for chunk in adaptive_rag.stream(inputs2, config2, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            print(f"\n--- 노드: {node_name} ---")
            if "generation" in node_output:
                final_answer2 = node_output["generation"]
except GraphRecursionError as e:
    print(f"\n⚠️ 재귀 한계 도달: {e}")
    # → 웹 검색이 관련 문서를 찾지 못해 루프에 빠졌어요. 질문을 구체화하거나 recursion_limit을 조정하세요.

if final_answer2:
    # ============================================================
    # 최종 답변:
    print(final_answer2)

## 10. invoke()로 전체 실행 및 결과 확인

`stream()`이 아닌 `invoke()`를 사용하면 최종 상태만 반환해요. 중간 과정을 보지 않고 최종 답변만 원할 때 사용해요.

In [ ]:
# ---------------------------------------------------
# invoke()로 최종 결과만 받기
# ---------------------------------------------------
# stream()은 중간 과정을 보여주고, invoke()는 최종 상태를 반환해요

config3 = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": str(uuid.uuid4())}
)

# 구글이 앤스로픽에 투자한 내용은 PDF에 있어요
query = "구글이 앤스로픽에 투자한 금액은 얼마인가요?"

# invoke()로 실행 - 최종 GraphState 반환
final_state = adaptive_rag.invoke(
    {"question": query},
    config3
)

print(f"질문: {final_state['question']}")
print(f"\n최종 답변:")
print(final_state["generation"])
print(f"\n사용된 문서 수: {len(final_state.get('documents', []))}개")

## 11. 실습: 라우팅 전략 직접 테스트

다양한 질문으로 어떤 경로가 선택되는지 확인해볼게요.

> 🎯 **강의 포인트**: 학생들이 각자 다른 질문을 입력하여 라우팅 결정이 어떻게 달라지는지 비교해보세요. 특히 "경계선" 질문 (2023년과 2024년의 경계에 있는 주제)이 어떻게 처리되는지 관찰해요.

In [ ]:
# ============================================================
# 실습 해설: 다양한 질문으로 라우팅 전략 테스트
# 힌트: 아래 질문 리스트를 수정하거나 새로운 질문을 추가해보세요
# 예상 결과:
#   - PDF 내용: route_question → vectorstore
#   - 최신 뉴스: route_question → web_search
# ============================================================

# 테스트할 질문 리스트 (직접 수정해보세요!)
test_questions = [
    "AI Brief 에서 언급된 OpenAI의 주요 발표는?",      # PDF에 있는 정보
    "최근 AI 규제 법안 현황은?",                        # 최신 정보 (웹 검색)
    # "앤스로픽의 Claude 모델 역사는?",                 # 주석 해제해서 테스트
]

# ============================================================
# 라우팅 전략 테스트
# ============================================================

for q in test_questions:
    print(f"\n질문: {q}")

    # 라우팅 결정만 먼저 확인 (전체 실행 전 라우팅 예측)
    route_result = question_router.invoke({"question": q})
    print(f"라우팅 결정: {route_result.datasource}")

    # 전체 실행
    test_config = RunnableConfig(
        recursion_limit=15,
        configurable={"thread_id": str(uuid.uuid4())}
    )

    try:
        result = adaptive_rag.invoke({"question": q}, test_config)
        answer = result.get("generation", "답변 없음")
        # 답변 미리보기 (100자)
        print(f"답변 미리보기: {answer[:100]}..." if len(answer) > 100 else f"답변: {answer}")
    except Exception as e:
        print(f"오류: {e}")

    # ----------------------------------------

## 12. 3가지 RAG 전략 비교

지금까지 배운 세 가지 전략을 정리해볼게요.

| 특성 | CRAG | Self-RAG | Adaptive RAG |
|------|------|----------|--------------|
| 검색 소스 | 벡터스토어 → 실패 시 웹 | 벡터스토어만 | 라우팅으로 사전 결정 |
| 문서 평가 | O (필터링) | O (필터링) | O (필터링) |
| 환각 검증 | X | O | O |
| 답변 관련성 | X | O | O |
| 자기 루프 | 1회 (재작성+웹검색) | 반복 가능 | 반복 가능 |
| 최신 정보 처리 | 사후 보완 | 불가 | 사전 라우팅 |
| 복잡도 | 중 | 중-상 | 상 |

### 프로덕션 환경에서의 선택 가이드

> 💡 **실무 팁**: 어떤 전략을 쓸지는 "LLM 호출 비용 vs 답변 품질" 트레이드오프로 결정해요.

| 시나리오 | 추천 전략 | 이유 |
|---------|----------|------|
| 사내 문서 Q&A (단일 소스) | **CRAG** | 웹 검색이 좋은 폴백이 됨 |
| 의료/법률 (높은 정확도 필수) | **Self-RAG** | 환각 검증이 핵심 |
| 복합 소스 (PDF + 웹 + DB) | **Adaptive RAG** | 사전 라우팅이 비용 절감 |
| MVP/프로토타입 | **Naive RAG** | 빠른 구현, 나중에 업그레이드 |
| 비용 민감한 서비스 | **CRAG** | LLM 호출 횟수가 적음 |

> 🔑 **핵심 개념**: Adaptive RAG는 CRAG와 Self-RAG의 장점을 결합해요:
> - CRAG의 "웹 검색 보강" + Self-RAG의 "생성 품질 검증" + 새로운 "사전 라우팅"
> - 세 전략 중 가장 강력하지만 LLM 호출 횟수가 가장 많아 비용도 높아요

> ⚠️ **자주 하는 실수**: 처음부터 Adaptive RAG를 적용하려 하지 마세요. Naive RAG → CRAG → Self-RAG → Adaptive RAG 순서로 점진적으로 개선하면서, LangSmith로 각 단계의 품질을 측정하는 것이 가장 효과적이에요.

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Query Router**: `Literal["vectorstore", "web_search"]`와 `with_structured_output(RouteQuery)`로 질문을 두 경로 중 하나로 자동 분류해요. 시스템 프롬프트에서 벡터스토어 내용을 명확히 설명하는 것이 핵심이에요.
- **3단계 조건부 라우팅**: (1) 최초 라우팅 → (2) 문서 평가 후 분기 → (3) 생성 품질 검증의 세 조건부 엣지가 순서대로 동작해요.
- **Hallucination Check**: `hallucination_grader`(환각 검증)와 `answer_grader`(관련성 검증)를 순차적으로 실행하는 복합 조건부 엣지예요. 두 검증을 모두 통과해야 END로 진행해요.
- **MemorySaver**: 루프 구조가 있는 그래프에서 상태를 체크포인트로 저장해요. `thread_id`로 여러 대화를 구분할 수 있어요.
- **Adaptive RAG vs CRAG/Self-RAG**: Adaptive RAG는 세 전략 중 가장 완전한 형태로, 웹 사전 라우팅 + 문서 평가 + 생성 품질 검증을 모두 포함해요.

## 다음 노트북 예고

다음 `05-Retrieval.ipynb`에서는 **검색 자체를 더 똑똑하게** 만드는 기법을 배워요. 2-Step / Agentic / Hybrid RAG 세 가지 아키텍처, **쿼리 강화(Query Enhancement)**로 N개의 다양한 쿼리를 생성하는 방법, **MMR(Maximal Marginal Relevance)**로 결과의 다양성을 보장하는 방법을 다뤄요. 지금까지 만든 RAG 루프의 "검색" 부분을 한 단계 업그레이드해요.